# Building Fast Queries

### ToC?

Intro
Setup
Inventory Class
Finding a Laptop From the Id
Improving Id Lookups
Comparing the Performance
Two Laptop Promotion
Optimizing Laptop Promotion
Comparing Promotion Functions
Finding Laptops Within a Budget

### Intro 

In this project, we are stepping into the role of laptop company owners. Our objective is to develop a robust class designed to help our customers efficiently navigate our product lineup. To ensure a seamless user experience, we will implement data preprocessing techniques to optimize query performance and speed.

Our system will be capable of handling the following three core queries:

1.  **ID Lookup:** Given a specific laptop ID, the system will retrieve and display the corresponding product data.
2.  **Price Pair Matching:** Given a specific dollar amount, the system will determine if there are any two laptops whose combined prices equal that total.
3.  **Budget Filtering:** Given a maximum budget, the system will identify all laptops with a price less than or equal to that limit.

### Setup

Let's first set up our project by taking a look at our data.

In [1]:
import csv

with open('laptops.csv') as f:
    reader = csv.reader(f)
    rows = list(reader)
    header = rows[0]
    rows = rows[1:]
    
print(header)
for i in range(5):
    print(rows[i])

['Id', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price']
['6571244', 'Apple', 'MacBook Pro', 'Ultrabook', '13.3', 'IPS Panel Retina Display 2560x1600', 'Intel Core i5 2.3GHz', '8GB', '128GB SSD', 'Intel Iris Plus Graphics 640', 'macOS', '1.37kg', '1339']
['7287764', 'Apple', 'Macbook Air', 'Ultrabook', '13.3', '1440x900', 'Intel Core i5 1.8GHz', '8GB', '128GB Flash Storage', 'Intel HD Graphics 6000', 'macOS', '1.34kg', '898']
['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', '575']
['9722156', 'Apple', 'MacBook Pro', 'Ultrabook', '15.4', 'IPS Panel Retina Display 2880x1800', 'Intel Core i7 2.7GHz', '16GB', '512GB SSD', 'AMD Radeon Pro 455', 'macOS', '1.83kg', '2537']
['8550527', 'Apple', 'MacBook Pro', 'Ultrabook', '13.3', 'IPS Panel Retina Display 2560x1600', 'Intel Core i5 3.1GHz', '8GB', '256GB SSD',

### Inventory Class

Let's begin building our class. In the code below, we will start by using our class to view the header row of our data and calculate how many rows our data contains.

In [2]:
class Inventory():

    def __init__(self, csv_filename):
        with open(csv_filename) as f:
            reader = csv.reader(f)
            rows = list(reader)
        self.header = rows[0]         
        self.rows = rows[1:]
        for row in self.rows:         
            row[-1] = int(row[-1])

In [3]:
inventory = Inventory('laptops.csv')  
print(inventory.header)               
print(len(inventory.rows))

['Id', 'Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight', 'Price']
1303


### Finding a Laptop From the Id

With our class created, we are now ready to build our first query. This function will accept a laptop ID value as an argument, and return rest of the data corresponding with the laptop. If there is no laptop with the given ID, the function will return None.

In [4]:
class Inventory():                    
    
    def __init__(self, csv_filename):
        with open(csv_filename) as f: 
            reader = csv.reader(f)
            rows = list(reader)
        self.header = rows[0]        
        self.rows = rows[1:]
        for row in self.rows:              
            row[-1] = int(row[-1])
            
    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:               
            if row[0] == laptop_id:
                return row
        return None                         

In [5]:
inventory = Inventory('laptops.csv')  
print(inventory.get_laptop_from_id('3362737'))
print(inventory.get_laptop_from_id('3362736'))

['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', 575]
None


### Improving Id Lookups

Now that we've built the `.get_laptop_from_id()` function, we want to build the same thing, but this time implementing the preprossessing technique of using a dictionary instead of a set.

In [6]:
class Inventory():                    
    
    def __init__(self, csv_filename):
        with open(csv_filename) as f: 
            reader = csv.reader(f)
            rows = list(reader)
        self.header = rows[0]        
        self.rows = rows[1:]
        for row in self.rows:              
            row[-1] = int(row[-1])
        self.id_to_row = {}
        for row in self.rows:              
            self.id_to_row[row[0]] = row
            
    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:               
            if row[0] == laptop_id:
                return row
        return None 

    def get_laptop_from_id_fast(self, laptop_id):
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

In [7]:
inventory = Inventory('laptops.csv')  
print(inventory.get_laptop_from_id_fast('3362737'))
print(inventory.get_laptop_from_id_fast('3362736'))

['3362737', 'HP', '250 G6', 'Notebook', '15.6', 'Full HD 1920x1080', 'Intel Core i5 7200U 2.5GHz', '8GB', '256GB SSD', 'Intel HD Graphics 620', 'No OS', '1.86kg', 575]
None


### Comparing the Performance

We have now made to functions to answer our first query, the `.get_laptop_from_id()` function and `.get_laptop_from_id_fast()` function. We will now compare the two to confirm which is faster, using the time and random libraries. 

In [8]:
import time
import random

ids = [str(random.randint(1000000, 9999999)) for _ in range(10000)]

inventory = Inventory('laptops.csv')

total_time_no_dict = 0
for identifier in ids:
    start = time.time()
    inventory.get_laptop_from_id(identifier)
    end = time.time()
    total_time_no_dict += end - start

total_time_dict = 0
for identifier in ids:
    start = time.time()
    inventory.get_laptop_from_id_fast(identifier)
    end = time.time()
    total_time_dict += end - start

print(total_time_no_dict)
print(total_time_dict)

0.37281203269958496
0.0010094642639160156


### Two Laptop Promotion

In [9]:
class Inventory():                    
    
    def __init__(self, csv_filename):
        with open(csv_filename) as f: 
            reader = csv.reader(f)
            rows = list(reader)
        self.header = rows[0]        
        self.rows = rows[1:]
        for row in self.rows:              
            row[-1] = int(row[-1])
        self.id_to_row = {}
        for row in self.rows:              
            self.id_to_row[row[0]] = row
            
    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:               
            if row[0] == laptop_id:
                return row
        return None 

    def get_laptop_from_id_fast(self, laptop_id):
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

    def check_promotion_dollars(self, dollars):
        for row in self.rows:               
            if row[-1] == dollars:
                return True
        for price1 in self.rows:
            for price2 in self.rows:
                if price1[-1] + price2[-1] == dollars:
                    return True
        return False

In [10]:
inventory = Inventory('laptops.csv')  
print(inventory.check_promotion_dollars(1000))
print(inventory.check_promotion_dollars(442))

True
False


### Optimizing Laptop Promotion

In [11]:
class Inventory():                    
    
    def __init__(self, csv_filename):
        with open(csv_filename) as f: 
            reader = csv.reader(f)
            rows = list(reader)
        self.header = rows[0]        
        self.rows = rows[1:]
        for row in self.rows:              
            row[-1] = int(row[-1])
        self.id_to_row = {}
        for row in self.rows:              
            self.id_to_row[row[0]] = row
        self.prices = set()
        for row in self.rows:              
            self.prices.add(row[-1])
                
    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:               
            if row[0] == laptop_id:
                return row
        return None 

    def get_laptop_from_id_fast(self, laptop_id):
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

    def check_promotion_dollars(self, dollars):
        for row in self.rows:               
            if row[-1] == dollars:
                return True
        for price1 in self.rows:
            for price2 in self.rows:
                if price1[-1] + price2[-1] == dollars:
                    return True
        return False

    def check_promotion_dollars_fast(self, dollars):
        if dollars in self.prices:
            return True
        for price in self.prices:                    
            if dollars - price in self.prices:
                return True
        return False 

In [12]:
inventory = Inventory('laptops.csv')  
print(inventory.check_promotion_dollars_fast(1000))
print(inventory.check_promotion_dollars_fast(442))

True
False


### Comparing Promotion Functions

In [13]:
prices = [random.randint(100, 5000) for _ in range(100)]

inventory = Inventory('laptops.csv')

total_time_no_set = 0
for price in prices:
    start = time.time()
    inventory.check_promotion_dollars(price)
    end = time.time()
    total_time_no_set += end - start

total_time_set = 0
for price in prices:
    start = time.time()
    inventory.check_promotion_dollars_fast(price)
    end = time.time()
    total_time_set += end - start

print(total_time_no_set)
print(total_time_set)

0.4362833499908447
0.0010001659393310547


### Finding Laptops Within a Budget

In [14]:
def row_price(row):
    return row[-1]

class Inventory():                    
    
    def __init__(self, csv_filename):
        with open(csv_filename) as f: 
            reader = csv.reader(f)
            rows = list(reader)
        self.header = rows[0]        
        self.rows = rows[1:]
        for row in self.rows:              
            row[-1] = int(row[-1])
        self.id_to_row = {}
        for row in self.rows:              
            self.id_to_row[row[0]] = row
        self.prices = set()
        for row in self.rows:              
            self.prices.add(row[-1])
        self.rows_by_price = sorted(self.rows, key=row_price)
                
    def get_laptop_from_id(self, laptop_id):
        for row in self.rows:               
            if row[0] == laptop_id:
                return row
        return None 

    def get_laptop_from_id_fast(self, laptop_id):
        if laptop_id in self.id_to_row:
            return self.id_to_row[laptop_id]
        return None

    def check_promotion_dollars(self, dollars):
        for row in self.rows:               
            if row[-1] == dollars:
                return True
        for price1 in self.rows:
            for price2 in self.rows:
                if price1[-1] + price2[-1] == dollars:
                    return True
        return False

    def check_promotion_dollars_fast(self, dollars):
        if dollars in self.prices:
            return True
        for price in self.prices:                    
            if dollars - price in self.prices:
                return True
        return False

    def find_laptop_with_price(self, target_price):
        range_start = 0                                   
        range_end = len(self.rows_by_price) - 1                       
        while range_start < range_end:
            range_middle = (range_end + range_start) // 2  
            price = self.rows_by_price[range_middle][-1]
            if price == target_price:                            
                return range_middle                        
            elif price < target_price:                           
                range_start = range_middle + 1             
            else:                                          
                range_end = range_middle - 1 
        price = self.rows_by_price[range_start][-1]
        if price != target_price:                  
            return -1                                      
        return range_start

    def find_first_laptop_more_expensive(self, target_price):
        range_start = 0
        range_end = len(self.rows_by_price) - 1
        while range_start < range_end:
            range_middle = (range_end + range_start) // 2
            price = self.rows_by_price[range_middle][-1]
            if price > target_price:
                range_end = range_middle
            else:
                range_start = range_middle + 1
        if self.rows_by_price[range_start][-1] <= target_price:                  
            return -1                                   
        return range_start

In [15]:
inventory = Inventory('laptops.csv')  
print(inventory.find_first_laptop_more_expensive(1000))
print(inventory.find_first_laptop_more_expensive(10000))

683
-1


### Conclusion